In [1]:
import numpy as np
import pandas as pd
import os

In [26]:
def load_table(name: str) -> pd.DataFrame:
    path = os.path.join(RAW_DIR, f"{name}.csv")
    df = pd.read_csv(path, low_memory=False)
    print(f"[{name}] {len(df)} satır, {len(df.columns)} sütun")
    return df


def check_key_uniqueness(df: pd.DataFrame, column: str, table_name: str) -> bool:
    n_total, n_unique = len(df), df[column].nunique()
    is_unique = n_unique == n_total
    status = "PK adayı OK" if is_unique else "DİKKAT: tekil değil!"
    print(f"[{table_name}] {column}: {n_unique}/{n_total} benzersiz ({status})")
    return is_unique


data = {name: load_table(name) for name in TABLES}

check_key_uniqueness(data["objects"], "id", "objects")
check_key_uniqueness(data["funding_rounds"], "id", "funding_rounds")

match_count = data["investments"]["funding_round_id"].isin(data["funding_rounds"]["id"]).sum()
total = len(data["investments"])
print(f"[investments] funding_round_id -> funding_rounds.id eşleşmesi: {match_count}/{total}")

for name, df in data.items():
    df.to_csv(os.path.join(PROCESSED_DIR, f"{name}_step1.csv"), index=False)

[objects] 462651 satır, 40 sütun
[funding_rounds] 52928 satır, 23 sütun
[investments] 80902 satır, 6 sütun
[funds] 1564 satır, 11 sütun
[acquisitions] 9562 satır, 12 sütun
[ipos] 1259 satır, 13 sütun
[objects] id: 462651/462651 benzersiz (PK adayı OK)
[funding_rounds] id: 52928/52928 benzersiz (PK adayı OK)
[investments] funding_round_id -> funding_rounds.id eşleşmesi: 80902/80902


In [28]:
def check_missing_values(df, table_name):
    missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
    print(f"[{table_name}] Eksik değer oranları:\n{missing_pct[missing_pct > 0]}")
    return missing_pct

def remove_duplicates(df, subset, table_name):
    before = len(df)
    df = df.drop_duplicates(subset=subset)
    print(f"[{table_name}] {before - len(df)} duplike satır silindi.")
    return df

def validate_dates(df, date_cols, table_name):
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            invalid = df[col].isna().sum()
            print(f"[{table_name}] {col}: {invalid} geçersiz/boş tarih")
    return df

def flag_invalid_amounts(df, amount_col, table_name):
    df["is_invalid_amount"] = (df[amount_col] <= 0) | df[amount_col].isna()
    print(f"[{table_name}] {df['is_invalid_amount'].sum()} satırda geçersiz/eksik tutar")
    return df

def flag_outliers_iqr(df, column, table_name):
    valid = df[column].dropna()
    q1, q3 = valid.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    df["is_outlier"] = (df[column] < lower) | (df[column] > upper)
    print(f"[{table_name}] {df['is_outlier'].sum()} aykırı değer (ham IQR)")
    return df

def flag_outliers_log_iqr(df, column, table_name):
    log_col = np.log1p(df[column].clip(lower=0))
    q1, q3 = log_col.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    df["is_outlier_log"] = (log_col < lower) | (log_col > upper)
    print(f"[{table_name}] {df['is_outlier_log'].sum()} aykırı değer (log-IQR)")
    return df

In [30]:
# funding_rounds temizleme
funding = pd.read_csv(os.path.join(PROCESSED_DIR, "funding_rounds_step1.csv"), low_memory=False)
check_missing_values(funding, "funding_rounds")
funding = remove_duplicates(funding, "id", "funding_rounds")
funding = validate_dates(funding, ["funded_at", "created_at"], "funding_rounds")
funding = flag_invalid_amounts(funding, "raised_amount_usd", "funding_rounds")
funding = flag_outliers_iqr(funding, "raised_amount_usd", "funding_rounds")
funding = flag_outliers_log_iqr(funding, "raised_amount_usd", "funding_rounds")
funding.to_csv(os.path.join(PROCESSED_DIR, "funding_rounds_cleaned.csv"), index=False)

[funding_rounds] Eksik değer oranları:
funded_at                    0.5
raised_currency_code         5.8
pre_money_currency_code     49.2
post_money_currency_code    42.5
source_url                  23.7
source_description          17.9
created_by                   8.8
dtype: float64
[funding_rounds] 0 duplike satır silindi.
[funding_rounds] funded_at: 248 geçersiz/boş tarih
[funding_rounds] created_at: 0 geçersiz/boş tarih
[funding_rounds] 6000 satırda geçersiz/eksik tutar
[funding_rounds] 5400 aykırı değer (ham IQR)
[funding_rounds] 6054 aykırı değer (log-IQR)


In [32]:
# acquisitions temizleme
acquisitions = pd.read_csv(os.path.join(PROCESSED_DIR, "acquisitions_step1.csv"), low_memory=False)
check_missing_values(acquisitions, "acquisitions")
acquisitions = validate_dates(acquisitions, ["acquired_at"], "acquisitions")
acquisitions = flag_invalid_amounts(acquisitions, "price_amount", "acquisitions")
acquisitions.to_csv(os.path.join(PROCESSED_DIR, "acquisitions_cleaned.csv"), index=False)

[acquisitions] Eksik değer oranları:
term_code             80.1
acquired_at            0.3
source_url            10.4
source_description    10.2
dtype: float64
[acquisitions] acquired_at: 29 geçersiz/boş tarih
[acquisitions] 6963 satırda geçersiz/eksik tutar


In [34]:
# ipos temizleme
ipos = pd.read_csv(os.path.join(PROCESSED_DIR, "ipos_step1.csv"), low_memory=False)
ipos = validate_dates(ipos, ["public_at"], "ipos")
ipos = flag_invalid_amounts(ipos, "valuation_amount", "ipos")
ipos.to_csv(os.path.join(PROCESSED_DIR, "ipos_cleaned.csv"), index=False)

[ipos] public_at: 600 geçersiz/boş tarih
[ipos] 1151 satırda geçersiz/eksik tutar


In [36]:
# objects temizleme
objects = pd.read_csv(os.path.join(PROCESSED_DIR, "objects_step1.csv"), low_memory=False)
check_missing_values(objects, "objects")
objects = remove_duplicates(objects, "id", "objects")
objects = validate_dates(objects, ["founded_at", "first_funding_at", "last_funding_at"], "objects")
objects["funding_total_usd"] = pd.to_numeric(objects["funding_total_usd"], errors="coerce")
invalid_total = (objects["funding_total_usd"] < 0).sum()
print(f"[objects] {invalid_total} satırda negatif funding_total_usd")
objects.to_csv(os.path.join(PROCESSED_DIR, "objects_cleaned.csv"), index=False)

[objects] Eksik değer oranları:
parent_id              94.0
category_code          73.4
founded_at             78.3
closed_at              99.4
domain                 62.2
homepage_url           62.2
twitter_username       72.7
logo_url               54.9
short_description      98.4
description            79.5
overview               49.2
tag_list               77.0
country_code           79.5
state_code             88.2
city                   80.4
first_investment_at    96.3
last_investment_at     96.3
first_funding_at       93.2
last_funding_at        93.2
first_milestone_at     78.3
last_milestone_at      78.3
created_by             26.6
dtype: float64
[objects] 0 duplike satır silindi.
[objects] founded_at: 362210 geçersiz/boş tarih
[objects] first_funding_at: 431144 geçersiz/boş tarih
[objects] last_funding_at: 431144 geçersiz/boş tarih
[objects] 0 satırda negatif funding_total_usd


In [38]:
only_raw = funding[(funding["is_outlier"] == True) & (funding["is_outlier_log"] == False)]
only_log = funding[(funding["is_outlier"] == False) & (funding["is_outlier_log"] == True)]
both = funding[(funding["is_outlier"] == True) & (funding["is_outlier_log"] == True)]

print(f"Sadece ham IQR'de aykırı: {len(only_raw)}")
print(f"Sadece log-IQR'de aykırı: {len(only_log)}")
print(f"İkisinde de aykırı: {len(both)}")

print("\nSadece log-IQR'de aykırı olanların tutar dağılımı:")
print(only_log["raised_amount_usd"].describe())

print("\nSadece ham IQR'de aykırı olanların tutar dağılımı:")
print(only_raw["raised_amount_usd"].describe())

Sadece ham IQR'de aykırı: 5388
Sadece log-IQR'de aykırı: 6042
İkisinde de aykırı: 12

Sadece log-IQR'de aykırı olanların tutar dağılımı:
count    6042.000000
mean        7.983946
std        98.151047
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      1700.000000
Name: raised_amount_usd, dtype: float64

Sadece ham IQR'de aykırı olanların tutar dağılımı:
count    5.388000e+03
mean     4.702763e+07
std      6.397471e+07
min      1.638900e+07
25%      2.031312e+07
50%      2.839177e+07
75%      4.500000e+07
max      9.500000e+08
Name: raised_amount_usd, dtype: float64


In [40]:
# Örnek satırlarla görsel doğrulama
sample = only_raw.sort_values("raised_amount_usd", ascending=False).head(15)
print(sample[["object_id", "raised_amount_usd", "funding_round_type", "funded_at"]])

      object_id  raised_amount_usd funding_round_type  funded_at
19173   c:11391        950000000.0          series-c+ 2011-01-11
8681    c:13219        920000000.0           post-ipo 2009-11-24
25961   c:39799        770000000.0              other 2010-11-30
44510   c:81750        750000000.0              other 2013-09-17
43786   c:25526        743000000.0              other 2013-09-09
35174    c:4175        725000000.0              other 2013-05-08
32602   c:65045        700000000.0     private-equity 2013-02-19
9898    c:38902        691000000.0            venture 2009-12-29
25543  c:130940        688000000.0            venture 2011-09-16
44148  c:202372        681759114.0           series-a 2011-12-28
45932  c:268869        674598700.0     private-equity 2011-09-30
26357  c:152173        625000000.0            venture 2012-03-16
34318  c:207187        586000000.0     private-equity 2013-04-29
51838  c:232638        570000000.0              other 2013-12-02
17338   c:33440        56

In [42]:
DISCREPANCY_THRESHOLD_USD = 1000

def reconcile_funding_totals(objects_df, funding_df, exclude_col=None):
    valid = funding_df[funding_df["is_invalid_amount"] == False]
    if exclude_col:
        valid = valid[valid[exclude_col] == False]

    computed = (
        valid.groupby("object_id")["raised_amount_usd"]
        .sum().reset_index()
        .rename(columns={"raised_amount_usd": "computed_funding_total"})
    )
    merged = objects_df[["id", "name", "funding_total_usd", "funding_rounds"]].merge(
        computed, left_on="id", right_on="object_id", how="left"
    )
    merged["computed_funding_total"] = merged["computed_funding_total"].fillna(0)
    merged["discrepancy"] = (merged["funding_total_usd"] - merged["computed_funding_total"]).abs()

    flagged = merged[merged["discrepancy"] > DISCREPANCY_THRESHOLD_USD].sort_values(
        "discrepancy", ascending=False
    )
    label = exclude_col if exclude_col else "baseline"
    print(f"({label}) {len(flagged)} şirkette tutarsızlık bulundu.")
    return flagged


objects_df = pd.read_csv(os.path.join(PROCESSED_DIR, "objects_cleaned.csv"), low_memory=False)
funding_df = pd.read_csv(os.path.join(PROCESSED_DIR, "funding_rounds_cleaned.csv"), low_memory=False)

reconcile_funding_totals(objects_df, funding_df)
reconcile_funding_totals(objects_df, funding_df, exclude_col="is_outlier")
flagged_log = reconcile_funding_totals(objects_df, funding_df, exclude_col="is_outlier_log")

flagged_log.to_csv(os.path.join(PROCESSED_DIR, "funding_discrepancies.csv"), index=False)

(baseline) 0 şirkette tutarsızlık bulundu.
(is_outlier) 3778 şirkette tutarsızlık bulundu.
(is_outlier_log) 25 şirkette tutarsızlık bulundu.
